# NYC 311 Service Requests — Data Cleaning Pipeline

**Author:** Riley Allen | rileyallen2412@gmail.com  
**GitHub:** [github.com/ryry2412](https://github.com/ryry2412)  
**LinkedIn:** [linkedin.com/in/rileyallen2412](https://linkedin.com/in/rileyallen2412)  
**Dataset:** [NYC 311 Service Requests (2010–Present)](https://data.cityofnewyork.us/Social-Services/311-Service-Requests-from-2010-to-Present/erm2-nwe9/about_data) — filtered Jan 2022 onward  
**Last updated:** 2025

---

## Project overview

This notebook demonstrates a **production-style data cleaning pipeline** applied to NYC's 311 Service Request dataset — one of the largest open civic datasets in the world, with millions of rows and a rich assortment of real-world data quality issues.

### Why this dataset?
NYC 311 data is an ideal cleaning challenge because it contains:
- **High missing value rates** across optional fields (e.g., cross streets, vehicle info)
- **Inconsistent text formatting** — borough names, agency codes, and complaint types entered by many different operators over many years
- **Date/time fields** stored as raw strings requiring parsing and validation
- **Logical inconsistencies** — closed dates before created dates, impossible response times
- **Duplicate records** from multi-agency routing
- **Rich opportunity for feature engineering** once the base data is clean

### Pipeline stages
| Stage | Description |
|-------|-------------|
| 1 | Load & inspect — understand raw data shape, types, nulls |
| 2 | Missing value analysis — visualize and make imputation decisions |
| 3 | Data type corrections — parse dates, cast numerics |
| 4 | Text normalization — standardize categoricals |
| 5 | Duplicate detection & removal |
| 6 | Feature engineering — response time, temporal features |
| 7 | Cleaning summary report — before/after comparison |

### Skills demonstrated
`Python` · `Pandas` · `NumPy` · `Matplotlib` · `Seaborn` · `data wrangling` · `documentation`

---
## Setup — imports and configuration

In [ ]:
# Standard library
import warnings
warnings.filterwarnings('ignore')

# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.float_format', '{:.2f}'.format)

# Plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 5)

print('Libraries loaded successfully.')

---
## Stage 1 — Load & inspect

**Goal:** Load the raw CSV and get a comprehensive first look at the dataset's shape, column types, and null rates before touching anything.

**Why this matters:** Skipping the inspection phase leads to uninformed cleaning decisions. We document every observation here so that downstream choices are traceable and justifiable.

In [ ]:
# Load raw data
# Source: NYC Open Data Socrata API — 200,000 rows, Jan 2022 onward
# API endpoint used to download:
#   https://data.cityofnewyork.us/resource/erm2-nwe9.csv
#     ?$limit=200000&$where=created_date>='2022-01-01'&$order=created_date DESC
#
# The CSV is saved locally as data/311_service_requests.csv and excluded
# from GitHub via .gitignore. See README for full reproduction instructions.
RAW_PATH = '../data/311_service_requests.csv'

df_raw = pd.read_csv(
    RAW_PATH,
    low_memory=False,   # prevents mixed-type warnings on large files
    dtype=str           # load everything as string first — we'll cast intentionally in Stage 3
)

print('Raw dataset loaded.')
print(f'  Source: NYC Open Data API (200k row sample, 2022–present)')
print(f'  Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
print(f'  Memory usage: {df_raw.memory_usage(deep=True).sum() / 1e6:.1f} MB')

In [ ]:
# Snapshot the initial shape for the final summary report (Stage 7)
# We track these metrics before any cleaning so we can show a before/after comparison
initial_shape = df_raw.shape
initial_nulls = df_raw.isnull().sum().sum()
initial_dupes = df_raw.duplicated().sum()

print('=== Baseline snapshot (pre-cleaning) ===')
print(f'  Rows:             {initial_shape[0]:>10,}')
print(f'  Columns:          {initial_shape[1]:>10,}')
print(f'  Total null cells: {initial_nulls:>10,}')
print(f'  Exact duplicates: {initial_dupes:>10,}')

In [ ]:
# First look at structure
# .info() gives us column names, non-null counts, and inferred dtypes
# Since we loaded everything as str, all dtypes show 'object' — that's expected
df_raw.info(verbose=True, show_counts=True)

In [ ]:
# Sample rows — visual sanity check
# Transposing makes wide datasets much easier to read
df_raw.head(3).T

In [ ]:
# Null rate per column — sorted descending
# This tells us which columns are mostly empty and may need to be dropped
null_rates = (
    df_raw.isnull().sum()
    .div(len(df_raw))
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
    .rename('null_pct')
    .reset_index()
    .rename(columns={'index': 'column'})
)

print('Top 20 columns by null rate:')
print(null_rates.head(20).to_string(index=False))

In [ ]:
# Drop columns with >80% null values — these carry too little signal to be useful
# Decision rationale: a column missing in 80%+ of rows cannot be reliably imputed
# and would add noise rather than value to any downstream analysis
HIGH_NULL_THRESHOLD = 80.0

cols_to_drop = null_rates.loc[
    null_rates['null_pct'] >= HIGH_NULL_THRESHOLD, 'column'
].tolist()

print(f'Columns with ≥{HIGH_NULL_THRESHOLD}% nulls (will be dropped):')
for col in cols_to_drop:
    pct = null_rates.loc[null_rates['column'] == col, 'null_pct'].values[0]
    print(f'  {col:<45} {pct:.1f}% null')

df = df_raw.drop(columns=cols_to_drop).copy()
print(f'\nColumns remaining after drop: {df.shape[1]}')

---
## Stage 2 — Missing value analysis

**Goal:** Visualize the null pattern across remaining columns and make deliberate, documented decisions about how to handle each group of missing values.

**Decision framework used:**
- **Drop the column** — >80% null (handled in Stage 1) or structurally irrelevant
- **Drop the row** — null in a critical identifier column (e.g., `Unique Key`, `Created Date`)
- **Fill with sentinel** — null is meaningful (e.g., 'UNKNOWN' for optional location fields)
- **Leave as-is** — null is genuinely not applicable (e.g., vehicle info for non-vehicle complaints)

In [ ]:
# Null heatmap — visually shows which columns have missing data and where
# Using a sample for performance on large datasets
SAMPLE_N = min(5000, len(df))
sample = df.sample(SAMPLE_N, random_state=42)

# Identify columns that still have any nulls
cols_with_nulls = df.columns[df.isnull().any()].tolist()

fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(
    sample[cols_with_nulls].isnull(),
    cbar=False,
    cmap=['#f0f0f0', '#d62728'],   # gray = present, red = missing
    yticklabels=False,
    ax=ax
)
ax.set_title('Missing value heatmap (5,000-row sample) — red = null', fontsize=13, pad=12)
ax.set_xlabel('')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig('../data/fig_null_heatmap.png', bbox_inches='tight')
plt.show()
print('Figure saved to data/fig_null_heatmap.png')

In [ ]:
# Bar chart of null rates for columns that still have any missing values
remaining_nulls = (
    df[cols_with_nulls].isnull().mean().mul(100).sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(14, 5))
remaining_nulls.plot(kind='bar', ax=ax, color='#d62728', edgecolor='white', width=0.7)
ax.axhline(y=50, color='#333', linestyle='--', linewidth=0.8, label='50% threshold')
ax.set_title('Null rate by column (remaining columns after Stage 1 drop)', fontsize=13, pad=12)
ax.set_ylabel('% null')
ax.set_xlabel('')
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.legend()
plt.tight_layout()
plt.savefig('../data/fig_null_rates.png', bbox_inches='tight')
plt.show()

In [ ]:
# --- Handling decision 1: Drop rows with null critical identifiers ---
# 'Unique Key' and 'Created Date' are essential; a row without them
# cannot be meaningfully identified or placed in time.
critical_cols = ['Unique Key', 'Created Date']

rows_before = len(df)
df = df.dropna(subset=critical_cols)
rows_after = len(df)

print(f'Rows dropped (null critical identifier): {rows_before - rows_after:,}')
print(f'Rows remaining: {rows_after:,}')

In [ ]:
# --- Handling decision 2: Fill optional location fields with 'UNKNOWN' ---
# Fields like 'Cross Street 1', 'Cross Street 2', 'Landmark' are optional at intake.
# Null here means 'not provided', not 'data error'. We sentinel-fill to preserve rows.
location_optional = [
    'Cross Street 1', 'Cross Street 2', 'Intersection Street 1',
    'Intersection Street 2', 'Landmark', 'Address Type'
]

# Only fill columns that actually exist in the dataset after Stage 1 drops
location_optional = [c for c in location_optional if c in df.columns]

df[location_optional] = df[location_optional].fillna('UNKNOWN')
print(f'Sentinel-filled {len(location_optional)} optional location columns with "UNKNOWN".')

In [ ]:
# --- Handling decision 3: Fill 'Closed Date' nulls with NaT (after parsing in Stage 3) ---
# Closed Date is null for open/unresolved requests — this is semantically valid.
# We will keep these nulls intentionally; they'll become NaT after datetime parsing.
# This is documented here so the decision is traceable.
print('Closed Date nulls: intentionally preserved (open requests have no close date).')
print(f'  Null count: {df["Closed Date"].isnull().sum():,} rows')

---
## Stage 3 — Data type corrections

**Goal:** Cast all columns to their correct data types. Since we loaded everything as `str` in Stage 1, we now make intentional casting decisions with proper error handling.

**Why load as string first?** Loading as string prevents Pandas from making silent, incorrect type assumptions on messy columns. We control every cast explicitly.

In [ ]:
# --- Date columns ---
# 'coerce' converts unparseable values to NaT instead of raising an error.
# This is safer than errors='raise' on civic data, which often has format inconsistencies.
date_columns = ['Created Date', 'Closed Date', 'Due Date', 'Resolution Action Updated Date']
date_columns = [c for c in date_columns if c in df.columns]

for col in date_columns:
    original_nulls = df[col].isnull().sum()
    df[col] = pd.to_datetime(df[col], errors='coerce')
    new_nulls = df[col].isnull().sum()
    coerced = new_nulls - original_nulls
    print(f'  {col:<45} parsed | {coerced:,} values coerced to NaT')

print('\nDate parsing complete.')

In [ ]:
# --- Numeric columns ---
# Latitude and Longitude should be float. 'X Coordinate' and 'Y Coordinate' are NYC state plane coords.
numeric_columns = ['Latitude', 'Longitude', 'X Coordinate (State Plane)', 'Y Coordinate (State Plane)']
numeric_columns = [c for c in numeric_columns if c in df.columns]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    print(f'  {col:<45} → float64 | nulls: {df[col].isnull().sum():,}')

print('\nNumeric casting complete.')

In [ ]:
# --- Validate coordinate ranges ---
# NYC bounding box (approximate):
#   Latitude:  40.48 to 40.92
#   Longitude: -74.28 to -73.68
# Rows outside this range are likely data entry errors and should be nullified

if 'Latitude' in df.columns and 'Longitude' in df.columns:
    invalid_coords = (
        (df['Latitude'] < 40.48) | (df['Latitude'] > 40.92) |
        (df['Longitude'] < -74.28) | (df['Longitude'] > -73.68)
    ) & df['Latitude'].notna()

    df.loc[invalid_coords, ['Latitude', 'Longitude']] = np.nan
    print(f'Invalid coordinate pairs nullified: {invalid_coords.sum():,}')

In [ ]:
# --- Logical date validation ---
# 'Closed Date' should never be before 'Created Date'.
# These are data entry errors — we null the Closed Date in such cases.

if 'Closed Date' in df.columns:
    invalid_dates = df['Closed Date'] < df['Created Date']
    df.loc[invalid_dates, 'Closed Date'] = pd.NaT
    print(f'Closed Date before Created Date (nullified): {invalid_dates.sum():,}')

---
## Stage 4 — Text normalization

**Goal:** Standardize all categorical text fields so that the same conceptual value is represented identically throughout the dataset.

**Why it matters:** 'BROOKLYN', 'Brooklyn', 'brooklyn', and 'Bklyn' all refer to the same borough, but Pandas treats them as 4 distinct values. This breaks groupbys, value counts, and any downstream analysis or dashboard.

In [ ]:
# Helper: show distinct values for a column before normalization
def inspect_categoricals(df, col, top_n=20):
    """Print value counts for a column to reveal inconsistencies."""
    print(f'--- {col} ({df[col].nunique()} unique values) ---')
    print(df[col].value_counts(dropna=False).head(top_n).to_string())
    print()

In [ ]:
# Inspect Borough before cleaning
if 'Borough' in df.columns:
    inspect_categoricals(df, 'Borough')

In [ ]:
# --- Step 1: Strip whitespace and standardize case across all object columns ---
# We apply this universally before any specific remapping
object_cols = df.select_dtypes(include='object').columns

for col in object_cols:
    df[col] = df[col].str.strip().str.upper()

print(f'Stripped whitespace and uppercased {len(object_cols)} text columns.')

In [ ]:
# --- Step 2: Borough normalization ---
# Common variants found in the dataset, mapped to clean canonical values
borough_map = {
    'BROOKLYN':          'BROOKLYN',
    'BRONX':             'BRONX',
    'THE BRONX':         'BRONX',
    'MANHATTAN':         'MANHATTAN',
    'NEW YORK':          'MANHATTAN',   # 'New York' used for Manhattan in some records
    'QUEENS':            'QUEENS',
    'STATEN ISLAND':     'STATEN ISLAND',
    'UNSPECIFIED':       'UNKNOWN',
    'N/A':               'UNKNOWN',
}

if 'Borough' in df.columns:
    df['Borough'] = df['Borough'].replace(borough_map)
    # Any value not in the map stays as-is; we'll catch residual unknowns below
    df['Borough'] = df['Borough'].fillna('UNKNOWN')
    print('After normalization:')
    inspect_categoricals(df, 'Borough')

In [ ]:
# --- Step 3: Status normalization ---
# Consolidate open/closed/pending variants
status_map = {
    'OPEN':              'OPEN',
    'ASSIGNED':          'OPEN',
    'IN PROGRESS':       'OPEN',
    'PENDING':           'OPEN',
    'CLOSED':            'CLOSED',
    'EMAIL SENT':        'CLOSED',
    'CANCELLED':         'CLOSED',
}

if 'Status' in df.columns:
    df['Status'] = df['Status'].replace(status_map).fillna('UNKNOWN')
    print('Status after normalization:')
    print(df['Status'].value_counts().to_string())

In [ ]:
# --- Step 4: Complaint Type — trim to top N + 'OTHER' ---
# There are hundreds of complaint type variants. For analysis we keep the top 30
# and bucket the rest as 'OTHER'. This makes groupby analysis tractable.

if 'Complaint Type' in df.columns:
    TOP_N = 30
    top_complaints = df['Complaint Type'].value_counts().nlargest(TOP_N).index
    df['Complaint Type'] = df['Complaint Type'].where(
        df['Complaint Type'].isin(top_complaints), other='OTHER'
    )
    print(f'Complaint Type: kept top {TOP_N}, bucketed rest as "OTHER".')
    print(f'Unique values now: {df["Complaint Type"].nunique()}')

---
## Stage 5 — Duplicate detection & removal

**Goal:** Identify and remove duplicate records. In 311 data, duplicates can arise from:
1. **Exact duplicates** — the same row ingested twice (system error)
2. **Near-duplicates** — same complaint, same address, same day, different `Unique Key` (multi-agency routing)

We handle both, with documented rationale for each decision.

In [ ]:
# --- Exact duplicates (all columns identical) ---
exact_dupes = df.duplicated().sum()
print(f'Exact duplicate rows: {exact_dupes:,}')

df = df.drop_duplicates()
print(f'Rows after exact deduplication: {len(df):,}')

In [ ]:
# --- Near-duplicates by Unique Key ---
# 'Unique Key' should be the primary identifier. If it appears more than once,
# those are system-level duplicates we should keep only the first occurrence of.

if 'Unique Key' in df.columns:
    key_dupes = df.duplicated(subset=['Unique Key']).sum()
    print(f'Rows with duplicate Unique Key: {key_dupes:,}')

    df = df.drop_duplicates(subset=['Unique Key'], keep='first')
    print(f'Rows after Unique Key deduplication: {len(df):,}')

In [ ]:
# --- Business-logic near-duplicates ---
# Definition: same complaint type, same address, same borough, same date
# These represent the same underlying citizen complaint routed to multiple agencies.
# We keep the first occurrence (earliest agency assignment) and drop the rest.

near_dupe_cols = ['Complaint Type', 'Incident Address', 'Borough', 'Created Date']
near_dupe_cols = [c for c in near_dupe_cols if c in df.columns]

# For date-level matching, extract date only (not time)
if 'Created Date' in df.columns:
    df['_created_date_only'] = df['Created Date'].dt.date
    near_dupe_cols_adj = [
        c if c != 'Created Date' else '_created_date_only'
        for c in near_dupe_cols
    ]
else:
    near_dupe_cols_adj = near_dupe_cols

near_dupe_cols_adj = [c for c in near_dupe_cols_adj if c in df.columns]
before = len(df)
df = df.drop_duplicates(subset=near_dupe_cols_adj, keep='first')

# Drop the helper column
if '_created_date_only' in df.columns:
    df = df.drop(columns=['_created_date_only'])

print(f'Business-logic near-duplicates removed: {before - len(df):,}')
print(f'Rows remaining: {len(df):,}')

---
## Stage 6 — Feature engineering

**Goal:** Extract analytically useful features from the cleaned data. This transforms a cleaning exercise into something actionable — these new columns are what an analyst or BI tool would actually use.

**Features we'll create:**
| New column | Source | Description |
|------------|--------|-------------|
| `response_hours` | Closed Date − Created Date | Time to close a request, in hours |
| `day_of_week` | Created Date | Day name (Monday–Sunday) |
| `hour_of_day` | Created Date | Hour 0–23 |
| `month` | Created Date | Month 1–12 |
| `season` | Created Date | Winter / Spring / Summer / Fall |
| `is_weekend` | Created Date | Boolean — Sat/Sun = True |

In [ ]:
# --- Response time in hours ---
if 'Closed Date' in df.columns and 'Created Date' in df.columns:
    df['response_hours'] = (
        (df['Closed Date'] - df['Created Date'])
        .dt.total_seconds()
        .div(3600)
        .round(2)
    )

    # Cap at 8760 hours (1 year) — anything beyond is likely a data error
    cap = 8760
    capped = (df['response_hours'] > cap).sum()
    df.loc[df['response_hours'] > cap, 'response_hours'] = np.nan
    print(f'response_hours created. Values capped (>{cap}h): {capped:,}')
    print(df['response_hours'].describe())

In [ ]:
# --- Temporal features from Created Date ---
if 'Created Date' in df.columns:
    df['day_of_week'] = df['Created Date'].dt.day_name()
    df['hour_of_day'] = df['Created Date'].dt.hour
    df['month']       = df['Created Date'].dt.month
    df['is_weekend']  = df['Created Date'].dt.dayofweek >= 5  # 5=Sat, 6=Sun

    # Season mapping
    def month_to_season(m):
        if m in [12, 1, 2]:  return 'WINTER'
        if m in [3, 4, 5]:   return 'SPRING'
        if m in [6, 7, 8]:   return 'SUMMER'
        return 'FALL'

    df['season'] = df['month'].map(month_to_season)

    print('Temporal features created: day_of_week, hour_of_day, month, is_weekend, season')
    print(df[['day_of_week', 'hour_of_day', 'month', 'is_weekend', 'season']].head(3))

In [ ]:
# --- Quick validation chart: complaints by hour of day ---
# This serves as a sanity check — we'd expect a peak during business hours
# and a dip overnight. Unexpected patterns would flag data quality issues.

if 'hour_of_day' in df.columns:
    fig, ax = plt.subplots(figsize=(12, 4))
    df['hour_of_day'].value_counts().sort_index().plot(
        kind='bar', ax=ax, color='#4e79a7', edgecolor='white', width=0.75
    )
    ax.set_title('311 complaints by hour of day (sanity check)', fontsize=13, pad=12)
    ax.set_xlabel('Hour of day (0 = midnight)')
    ax.set_ylabel('Number of complaints')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig('../data/fig_complaints_by_hour.png', bbox_inches='tight')
    plt.show()

In [ ]:
# --- Response time distribution by borough ---
if 'response_hours' in df.columns and 'Borough' in df.columns:
    valid = df[df['response_hours'].notna() & (df['Borough'] != 'UNKNOWN')]

    fig, ax = plt.subplots(figsize=(12, 5))
    sns.boxplot(
        data=valid,
        x='Borough',
        y='response_hours',
        order=sorted(valid['Borough'].unique()),
        palette='muted',
        showfliers=False,   # hide extreme outliers for readability
        ax=ax
    )
    ax.set_title('Response time (hours) by borough — outliers hidden', fontsize=13, pad=12)
    ax.set_xlabel('')
    ax.set_ylabel('Hours to close request')
    plt.tight_layout()
    plt.savefig('../data/fig_response_by_borough.png', bbox_inches='tight')
    plt.show()

---
## Stage 7 — Cleaning summary report

**Goal:** Produce a clear before/after comparison that documents exactly what this pipeline did and why. This is the most portfolio-important section — it demonstrates that you approach data cleaning as a professional process, not just code.

This summary should be included verbatim (or in slightly edited form) in the project README.

In [ ]:
# Collect final metrics
final_shape = df.shape
final_nulls = df.isnull().sum().sum()
final_dupes = df.duplicated().sum()

# Compute deltas
rows_removed    = initial_shape[0] - final_shape[0]
cols_removed    = initial_shape[1] - final_shape[1]
nulls_reduced   = initial_nulls - final_nulls
dupes_removed   = initial_dupes - final_dupes
new_features    = len([c for c in df.columns if c in [
    'response_hours', 'day_of_week', 'hour_of_day', 'month', 'is_weekend', 'season'
]])

print('=' * 58)
print('  NYC 311 DATA CLEANING PIPELINE — SUMMARY REPORT')
print('=' * 58)
print(f'{"Metric":<35} {"Before":>10} {"After":>10}')
print('-' * 58)
print(f'{"Rows":<35} {initial_shape[0]:>10,} {final_shape[0]:>10,}')
print(f'{"Columns":<35} {initial_shape[1]:>10,} {final_shape[1]:>10,}')
print(f'{"Total null cells":<35} {initial_nulls:>10,} {final_nulls:>10,}')
print(f'{"Exact duplicate rows":<35} {initial_dupes:>10,} {final_dupes:>10,}')
print('-' * 58)
print(f'{"Rows removed":<35} {rows_removed:>10,}')
print(f'{"Columns removed (>80% null)":<35} {cols_removed:>10,}')
print(f'{"Null cells reduced":<35} {nulls_reduced:>10,}')
print(f'{"Duplicates removed":<35} {dupes_removed:>10,}')
print(f'{"New engineered features":<35} {new_features:>10,}')
print('=' * 58)

In [ ]:
# Final column inventory — what we're left with and their dtypes
col_summary = pd.DataFrame({
    'dtype':    df.dtypes.astype(str),
    'null_pct': (df.isnull().mean() * 100).round(2),
    'n_unique': df.nunique()
}).reset_index().rename(columns={'index': 'column'})

print('Final dataset — column inventory:')
print(col_summary.to_string(index=False))

In [ ]:
# Export cleaned dataset
OUTPUT_PATH = '../data/311_cleaned.csv'
df.to_csv(OUTPUT_PATH, index=False)
print(f'Clean dataset exported to {OUTPUT_PATH}')
print(f'Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

---
## Cleaning decisions log

This section documents every substantive decision made in this pipeline. Good data cleaning is auditable — someone reading this notebook should be able to reproduce every choice.

| Decision | Rationale |
|----------|-----------|
| Loaded all columns as `str` initially | Prevents silent incorrect type inference on messy civic data |
| Dropped columns with ≥80% null values | Columns this sparse cannot be reliably imputed and add noise |
| Dropped rows with null `Unique Key` or `Created Date` | These are non-negotiable identifiers; a row without them is unusable |
| Filled optional location fields with `'UNKNOWN'` | Null = 'not provided by caller', not a data error; preserving the row is better than dropping |
| Preserved null `Closed Date` as `NaT` | Null = request still open; this is semantically correct, not a data error |
| Used `errors='coerce'` for date parsing | Safely handles format inconsistencies across years of data entry |
| Nullified coordinates outside NYC bounding box | Likely data entry errors; better to mark unknown than use wrong location |
| Nullified `Closed Date` < `Created Date` | Logically impossible; indicates a data entry error |
| Uppercased all text fields | Eliminates case-sensitivity false-distincts in groupby operations |
| Remapped borough variants to canonical names | 5 boroughs should produce 5 values, not dozens of variants |
| Bucketed rare complaint types as `'OTHER'` | Preserves top-30 signal without a long tail of noise |
| Capped response_hours at 8,760 (1 year) | Implausibly long times are data errors, not real resolution times |

---
*Project 3 of 4 · Riley Allen's Data Analytics Portfolio*  
*See also: [Project 1 — SQL](https://github.com/ryry2412/retail-ecommerce-sql) · [Project 2 — EDA & A/B Test](https://github.com/ryry2412/bank-marketing-eda-ab-test)*